# 07 | Alda-1: imagens reais e controles R1-R3

**Papel nesta entrega:** Principal: inspecao das imagens originais

**Origem:** https://colab.research.google.com/drive/1brwjVv8bTDOKEITls8hc_b1OAtf0Gknj

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Os seletores de retomada/abertura automatica foram colocados em estado seguro nesta copia. Ative a etapa pretendida somente apos ler o protocolo; as quatro R4 ja foram consultadas no historico do trabalho.

Para o TCC foram utilizados os tres campos 1, 5 e 9 de R1-R3: 36 pocos e 432 arquivos. O modulo visual opcional de R4 NAO integra os resultados visuais do manuscrito. A ficha de 12 resumos preenchida pelo revisor e uma fonte separada, nao uma predicao do caderno. A ficha detalhada produzida aqui foi posteriormente substituida para o resumo humano; nao invente novas respostas.

# Alda-1 nos quatro locais: imagens de R1–R3 e R4 opcional

**IMTM, FMP, MEDINA e USC | HepG2 | inspeção descritiva posterior à modelagem.**

No piloto observamos que o brilho e a ocupação celular variavam entre os locais. Um único campo de R1 não permite saber se essas diferenças se conservam entre campos ou réplicas. Este caderno amplia **somente Alda-1 (EOS100315)**, com os mesmos critérios de amostragem e com controles correspondentes.

**Pergunta:** a aparência da marcação de Mito em relação aos DMSO se repete em R1, R2 e R3? Uma mudança de brilho pode ser separada de mudanças de ocupação, fundo ou distribuição?

Não serão ajustados modelos, calculados novos escores, procurados candidatos, segmentadas mitocôndrias ou preenchidas conclusões biológicas automaticamente. Os resultados anteriores permanecem intactos.

**Como usar:** execute a seção 1, reinicie somente se solicitado e depois execute tudo. R4 está desligada na seção 13. Não reutilize como pasta de saída a execução do piloto.


## 1. Preparar o ambiente

Somente bibliotecas de leitura e visualização são necessárias. As bibliotecas já instaladas não são atualizadas deliberadamente. Se uma instalação ocorrer, o caderno interrompe para reiniciar a sessão antes das importações.

In [ ]:
import sys, subprocess, importlib.util, importlib.metadata
REQUISITOS = {'numpy':'numpy','pandas':'pandas','PIL':'Pillow','requests':'requests',
              'tifffile':'tifffile','imagecodecs':'imagecodecs'}
FALTANTES = [pkg for mod, pkg in REQUISITOS.items() if importlib.util.find_spec(mod) is None]
if FALTANTES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed', *FALTANTES])
    raise RuntimeError('INSTALACAO CONCLUIDA. Reinicie a sessao e repita esta celula.')
import numpy, pandas, PIL, requests, tifffile, imagecodecs
VERSOES = {'python':sys.version.split()[0], **{p:importlib.metadata.version(p) for p in REQUISITOS.values()}}
print('AMBIENTE PRONTO', VERSOES)


## 2. Ficha e caminhos da ampliação

O plano padrão tem **36 poços**: um Alda-1 e dois DMSO para cada combinação de quatro locais e três réplicas. Três campos por poço e quatro canais correspondem a **432 arquivos de imagem**, quando todas as correspondências forem válidas.

`RETOMAR_PASTA` deve apontar para uma pasta criada **por este caderno**. Os TIFFs do piloto podem ser lidos em cache, após conferir a chave remota, o ETag, o tamanho e o SHA256. Não são modificados nem copiados desnecessariamente. O limite é de 2.048 MiB para o plano R1–R3, e o volume real é mostrado antes do download.

In [ ]:
from pathlib import Path
from google.colab import drive
from IPython.display import display, Markdown, Image as IPyImage

drive.mount('/content/drive')
BASE_INSPECAO = Path('/content/drive/MyDrive/mba_tcc/comparacao_4_locais_HepG2/inspecao_biologica_4_locais')
RAIZ = BASE_INSPECAO / 'Alda1_R1_R3_R4'
FONTES_ZIP = RAIZ / 'fontes_inspecao_biologica.zip'
PILOTO = BASE_INSPECAO / 'resultados/inspecao_20261003T200334_744222Z'
PASTAS_CACHE = [PILOTO / 'imagens_originais']
RETOMAR_PASTA = ''  # Vazio: campanha nova. Retomada: caminho exato impresso por este caderno.
LIMITE_R1_R3_MiB = 2048


## 3. Funções do procedimento

Todo o código está nesta célula; não há dependência de um script externo. Ela define a leitura, as conferências, o cache, as faixas comuns, a ficha e o bloqueio de R4. Não executa downloads por si só.

A ligação com as imagens segue `barcode_platemap.csv`, o mapa dos poços e `load_data.csv`. Não se presume a identidade de um canal pelo número do arquivo ou pelo nome de uma placa. Se uma correspondência for ambígua, a etapa para e gera uma pendência.

In [ ]:
"""Alda-1 visual follow-up. No fits, new scores, masks, or candidate selection.
All web objects must come from cpg0036 in the public Cell Painting Gallery.
"""
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse, unquote, quote
import hashlib, io, json, re, time, zipfile, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageDraw, ImageFont
import tifffile

PROJECT='cpg0036-EU-OS-bioactives'
BUCKET='cellpainting-gallery'
HOST='https://cellpainting-gallery.s3.amazonaws.com'
SITES=['IMTM','FMP','MEDINA','USC']
CHANNELS=['DNA','ER','AGP','Mito']
REPS=['R1','R2','R3']
CASE_CODES=['EOS100315','EOS102190','EOS100964','EOS102355','EOS100147','EOS102419']

def require(ok,msg):
    if not bool(ok): raise ValueError(msg)

def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def jsave(p,x):
    Path(p).write_text(json.dumps(x,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

def safe_extract(path,dest):
    dest=Path(dest).resolve();dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(path) as z:
        for n in z.namelist():
            require((dest/n).resolve().is_relative_to(dest),'ZIP com caminho inseguro')
        z.extractall(dest)

def normwell(x):
    m=re.fullmatch(r'([A-Pa-p])0*([1-9]|1[0-9]|2[0-4])',str(x).strip())
    require(m is not None,'Poco fora do formato A01-P24: '+str(x))
    return m[1].upper()+f'{int(m[2]):02d}'

def canonical_plate(x):
    """Only explicit B1001_R1 / B1001R1 / R1B1001; not Bioactives_1_plate_1."""
    s=str(x)
    a=re.findall(r'(B\d{4})[_-]?(R[1-4])',s,re.I)
    b=[(v[1],v[0]) for v in re.findall(r'(R[1-4])[_-]?(B\d{4})',s,re.I)]
    hits=set((p.upper(),r.upper()) for p,r in a+b)
    require(len(hits)<=1,'Mais de uma placa/replica no identificador: '+s)
    return next(iter(hits)) if hits else None

def natural(x):
    return tuple((0,int(t)) if t.isdigit() else (1,t.lower()) for t in re.split(r'(\d+)',str(x)) if t)

def normfield(x):
    s=str(x).strip()
    if re.fullmatch(r'\d+(\.0+)?',s): return str(int(float(s)))
    require(bool(s),'Campo vazio no load_data')
    return s

def make_plan(sources,mode='piloto',codes=None,replicas=None):
    src=Path(sources)
    cases=pd.read_csv(src/'pocos_casos_R1_R3.csv')
    controls=pd.read_csv(src/'DMSO_R1_R3.csv')
    for d in [cases,controls]:
        require(set(d.replica)<=set(REPS),'R4 nao pertence a esta etapa')
        d['poco']=d.poco.map(normwell)
        require(not d.duplicated(['site','placa','replica','poco']).any(),'Poço repetido')
    if codes is None: codes=['EOS100315'] if mode=='piloto' else CASE_CODES
    if replicas is None: replicas=['R1'] if mode=='piloto' else REPS
    require(set(replicas)<=set(REPS),'Nao abrir R4 nesta etapa')
    d=cases[cases.composto.isin(codes)&cases.replica.isin(replicas)].copy()
    require(len(d)>0,'Nenhum caso disponivel para esta selecao')
    cs=[]
    for (s,p,r),g in d.groupby(['site','placa','replica']):
        q=controls[(controls.site==s)&(controls.placa==p)&(controls.replica==r)].copy()
        require(len(q)>=2,'Menos de dois DMSO em '+str((s,p,r)))
        q['ordem']=q.poco.map(lambda w:hashlib.sha256(f'20261003|{s}|{p}|{r}|{w}'.encode()).hexdigest())
        q=q.sort_values('ordem').head(2).drop(columns='ordem')
        q['tipo']='controle_DMSO';q['nome']='DMSO';cs.append(q)
    d=pd.concat([d]+cs,ignore_index=True)
    d['sample_id']=d.apply(lambda r:'S'+hashlib.sha256(f'{r.site}|{r.placa}|{r.replica}|{r.poco}'.encode()).hexdigest()[:12],axis=1)
    d['criterio_controles']='2 DMSO por placa/replica por ordem SHA256; sem usar intensidade, contagem ou escore'
    require(d.sample_id.is_unique,'Colisao de identificadores')
    return d.sort_values(['site','placa','replica','sample_id']).reset_index(drop=True)

class PublicGallery:
    def __init__(self,cache,timeout=45):
        self.cache=Path(cache);self.cache.mkdir(parents=True,exist_ok=True)
        self.s=requests.Session();self.timeout=timeout
    def request(self,method,url,**kwargs):
        for attempt in range(3):
            try:
                r=self.s.request(method,url,timeout=self.timeout,**kwargs)
                r.raise_for_status();return r
            except requests.RequestException:
                if attempt==2:raise
                time.sleep(2**attempt)
    def list(self,prefix,max_items=2000):
        require(prefix.startswith(PROJECT+'/'),'Prefixo nao autorizado')
        params={'list-type':'2','prefix':prefix,'max-keys':'1000'};out=[]
        while True:
            r=self.request('GET',HOST,params=params)
            root=ET.fromstring(r.content);ns={'s':'http://s3.amazonaws.com/doc/2006-03-01/'}
            for el in root.findall('s:Contents',ns):
                out.append({'key':el.findtext('s:Key',namespaces=ns),'bytes':int(el.findtext('s:Size',namespaces=ns)), 'etag':el.findtext('s:ETag',namespaces=ns)})
            require(len(out)<=max_items,'Listagem excedeu limite de seguranca; nenhuma imagem baixada')
            if root.findtext('s:IsTruncated',namespaces=ns)!='true':break
            token=root.findtext('s:NextContinuationToken',namespaces=ns)
            require(token,'Paginacao sem token');params['continuation-token']=token
        return out
    def bytes(self,key,max_bytes=32*1024*1024):
        p=self.cache/(hashlib.sha256(key.encode()).hexdigest()+'.bin')
        if p.exists():return p.read_bytes()
        info=self.head(key);require(info['bytes']<=max_bytes,'Objeto de metadados muito grande: '+key)
        r=self.request('GET',object_url(key),stream=True);buf=io.BytesIO();n=0
        for part in r.iter_content(1024*1024):
            n+=len(part);require(n<=max_bytes,'Transferencia excedeu limite');buf.write(part)
        blob=buf.getvalue();require(len(blob)==info['bytes'],'Transferencia incompleta');p.write_bytes(blob)
        return blob
    def head(self,key):
        r=self.request('HEAD',object_url(key));n=r.headers.get('Content-Length')
        require(n is not None,'HEAD sem tamanho: '+key)
        return {'bytes':int(n),'etag':r.headers.get('ETag',''),'last_modified':r.headers.get('Last-Modified','')}

def object_url(key):
    require(key.startswith(PROJECT+'/') and '..' not in Path(key).parts,'Chave nao permitida')
    return HOST+'/'+quote(key,safe='/')

def object_key(text,site):
    text=str(text).strip().replace('\\','/')
    require(bool(text) and text.lower()!='nan','URL vazia')
    if text.startswith('s3://'):
        u=urlparse(text);require(u.netloc==BUCKET,'Bucket diferente');key=unquote(u.path.lstrip('/'))
    elif text.startswith(('http://','https://')):
        u=urlparse(text);require(u.scheme=='https' and (u.netloc==BUCKET+'.s3.amazonaws.com' or u.netloc==BUCKET+'.s3.us-east-1.amazonaws.com'),'Host nao autorizado')
        key=unquote(u.path.lstrip('/'));require(not u.query,'URL com query inesperada')
    elif PROJECT+'/' in text:
        key=text[text.index(PROJECT+'/'):]
    else:raise ValueError('Caminho sem prefixo documental reconhecido: '+text)
    require(key.startswith(f'{PROJECT}/{site}/images/') and '..' not in Path(key).parts,'Imagem fora do local autorizado: '+key)
    return key

def read_remote_table(blob):
    # No executable content: read textual CSV/TSV only.
    return pd.read_csv(io.BytesIO(blob),sep=None,engine='python',dtype=str,keep_default_na=False)

def barcode_lookup(gallery,site):
    out={};records=[]
    keys=gallery.list(f'{PROJECT}/{site}/workspace/metadata/',max_items=1500)
    for ob in keys:
        k=ob['key']
        if 'hepg2' not in k.lower() or not k.lower().endswith('barcode_platemap.csv'):continue
        blob=gallery.bytes(k);d=read_remote_table(blob)
        require({'Assay_Plate_Barcode','Plate_Map_Name'}<=set(d),'barcode_platemap sem colunas esperadas')
        for _,r in d.iterrows():
            c=canonical_plate(r.Plate_Map_Name)
            if c is None:continue
            batch=k.split('/platemaps/')[-1].split('/')[0]
            key=(batch,r.Assay_Plate_Barcode)
            require(key not in out or out[key]==c,'Barcode ambiguo')
            out[key]=c
            map_candidates=[v['key'] for v in keys if v['key'].startswith(str(Path(k).parent)+'/platemap/') and Path(v['key']).stem==str(r.Plate_Map_Name) and Path(v['key']).suffix.lower() in ['.txt','.tsv','.csv']]
            records.append({'site':site,'barcode':r.Assay_Plate_Barcode,'batch':batch,'placa':c[0],'replica':c[1],'fonte':k,'sha256':hashlib.sha256(blob).hexdigest(),'platemap_key':map_candidates[0] if len(map_candidates)==1 else ''})
    return out,records

def validate_public_platemap(df,plan,site,identity,key):
    """Confirm well -> EOS/control and dose against the public map; no name matching."""
    def col(allowed):
        found=[v for v in allowed if v in df]
        require(len(found)==1,'Coluna do platemap ausente/ambigua: '+str(allowed))
        return found[0]
    wc=col(['well_position','Metadata_Well']);ec=col(['Metadata_EOS','compound_id'])
    dc=col(['Metadata_Concentration','Concentration'])
    d=df.copy();d['poco']=d[wc].map(normwell)
    require(not d.poco.duplicated().any(),'Poco duplicado no platemap publico')
    p,r=identity;g=plan[(plan.site==site)&(plan.placa==p)&(plan.replica==r)]
    mapped=d.set_index('poco');rows=[]
    for _,row in g.iterrows():
        require(row.poco in mapped.index,'Poco ausente do platemap publico: '+row.poco)
        v=mapped.loc[row.poco]
        require(str(v[ec]).strip()==str(row.composto),'Identidade do composto diverge do mapa publico: '+str((site,p,r,row.poco)))
        a,b=float(v[dc]),float(row.dose)
        require(np.isfinite(a) and a==b,'Concentracao numerica diverge do mapa publico; nao corrigir automaticamente')
        rows.append({'site':site,'placa':p,'replica':r,'poco':row.poco,'composto':row.composto,'dose_numerica':b,'platemap_key':key,'resultado':'identidade_e_dose_documentais_concordantes','nota':'Unidade numerica preservada; nao inferida a partir do nome do arquivo'})
    return rows

def infer_identity(key,aliases):
    explicit=canonical_plate(key)
    pieces=key.split('/');candidates=set()
    for (batch,barcode),v in aliases.items():
        if batch in pieces and barcode in pieces:candidates.add(v)
    if candidates:
        require(len(candidates)==1,'Mais de um barcode corresponde ao load_data')
        return next(iter(candidates)) # authority is barcode table; not microscope basename
    return explicit

def parse_load_data(df,site,identity,key,aliases,allowed_reps=tuple(REPS)):
    p,r=identity
    require(r in allowed_reps, 'Replica nao autorizada no leitor: '+str(r))
    require({'Metadata_Well','Metadata_Site'}<=set(df),'load_data sem Well/Site: '+key)
    rows=[];batch=key.split('/load_data_csv/')[-1].split('/')[0]
    for i,row in df.iterrows():
        if 'Metadata_Plate' in df and str(row.Metadata_Plate):
            raw=str(row.Metadata_Plate).strip();resolved=aliases.get((batch,raw)) or canonical_plate(raw)
            if resolved is None and raw.upper()==p:resolved=identity  # replica is documented in this load_data directory
            require(resolved is not None and resolved==identity,'Placa do load_data nao confere: '+key+' / '+raw)
        w=normwell(row.Metadata_Well);field=normfield(row.Metadata_Site)
        for ch in CHANNELS:
            cols=[c for c in ['URL_Orig'+ch,'URL_'+ch] if c in df and str(row[c]).strip()]
            if cols:
                ks={object_key(row[c],site) for c in cols};require(len(ks)==1,'URLs de canal conflitantes')
                k=next(iter(ks));source=cols[0]
            else:
                pairs=[('PathName_Orig'+ch,'FileName_Orig'+ch),('PathName_'+ch,'FileName_'+ch)]
                pairs=[v for v in pairs if v[0] in df and v[1] in df and row[v[1]]]
                require(len(pairs)==1,'Canal nao documentado sem ambiguidade: '+ch)
                pc,fc=pairs[0];k=object_key(row[pc].rstrip('/')+'/'+row[fc],site);source=pc+' + '+fc
            rows.append({'site':site,'placa':p,'replica':r,'poco':w,'campo':field,'canal':ch,'key':k,'fonte_load_data':key,'linha_load_data':i+2,'coluna_canal':source})
    result=pd.DataFrame(rows)
    cols=['site','placa','replica','poco','campo','canal']
    require(not result.duplicated(cols).any(),'Mais de uma imagem para o mesmo campo/canal: '+key)
    return result

def discover_images(plan,gallery,out,allowed_reps=tuple(REPS)):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    require(set(plan.replica)<=set(allowed_reps), 'Plano contem replica nao autorizada')
    allrows=[];audit=[];errors=[];barcodes=[];mapaudit=[]
    for site,g in plan.groupby('site'):
        wanted=set(zip(g.placa,g.replica));aliases,br=barcode_lookup(gallery,site);barcodes+=br
        objs=gallery.list(f'{PROJECT}/{site}/workspace/load_data_csv/',max_items=1500)
        keys=[x['key'] for x in objs if 'hepg2' in x['key'].lower() and x['key'].lower().endswith('.csv')]
        # Prefer current load_data.csv; do not touch load_data_csv_orig or use guessed channel numbers.
        keys=[k for k in keys if k.endswith('/load_data.csv') or k.endswith('/load_data_with_illum.csv')]
        matched={}
        for k in keys:
            ident=infer_identity(k,aliases)
            if ident not in wanted:continue
            matched.setdefault(ident,[]).append(k)
        for ident in sorted(wanted):
            try:
                candidates=matched.get(ident,[])
                base=[k for k in candidates if k.endswith('/load_data.csv')]
                if base:candidates=base
                require(len(candidates)==1,f'{site} {ident}: esperado um load_data, encontrados {len(candidates)}. Rever barcode, nao adivinhar.')
                mapkeys={v['platemap_key'] for v in br if (v['placa'],v['replica'])==ident and v.get('platemap_key')}
                require(len(mapkeys)==1,f'{site} {ident}: platemap publico unico nao resolvido; nao presumir EOS pelo nome da placa')
                mk=next(iter(mapkeys));mb=gallery.bytes(mk)
                checks=validate_public_platemap(read_remote_table(mb),plan,site,ident,mk)
                for chk in checks:chk['platemap_sha256']=hashlib.sha256(mb).hexdigest()
                mapaudit+=checks
                key=candidates[0];blob=gallery.bytes(key);df=read_remote_table(blob)
                table=parse_load_data(df,site,ident,key,aliases,allowed_reps=allowed_reps)
                table=table[table.poco.isin(g[(g.placa==ident[0])&(g.replica==ident[1])].poco)]
                allrows.append(table);audit.append({'site':site,'placa':ident[0],'replica':ident[1],'key':key,'sha256':hashlib.sha256(blob).hexdigest(),'n_linhas':len(df)})
            except Exception as e:errors.append({'site':site,'placa':ident[0],'replica':ident[1],'erro':str(e)})
    pd.DataFrame(audit).to_csv(out/'load_data_fontes.csv',index=False)
    pd.DataFrame(barcodes).to_csv(out/'barcodes_fontes.csv',index=False)
    pd.DataFrame(mapaudit).to_csv(out/'platemap_publico_conferencia.csv',index=False)
    pd.DataFrame(errors,columns=['site','placa','replica','erro']).to_csv(out/'pendencias_metadados.csv',index=False)
    require(not errors,'Correspondencia com imagens pendente; consulte pendencias_metadados.csv. Nenhum campo foi substituido.')
    require(allrows,'Nenhum load_data compativel encontrado')
    data=pd.concat(allrows,ignore_index=True)
    require(not data.duplicated(['site','placa','replica','poco','campo','canal']).any(),'Imagens duplicadas entre arquivos')
    return data

def choose_fields(data,plan,n_fields=1):
    require(n_fields>=1,'Numero de campos invalido');parts=[];audit=[]
    for (s,p,r),g in plan.groupby(['site','placa','replica']):
        t=data[(data.site==s)&(data.placa==p)&(data.replica==r)]
        sets=[]
        for w in g.poco:
            q=t[t.poco==w];ok=q.groupby('campo').canal.agg(set)
            fields=set(ok[ok.map(lambda a:a==set(CHANNELS))].index)
            require(fields,f'Nenhum campo completo de 4 canais: {s}/{p}/{r}/{w}')
            sets.append(fields)
        common=sorted(set.intersection(*sets),key=natural)
        require(len(common)>=n_fields,f'{s}/{p}/{r}: poucos campos comuns. Nao substituir campos silenciosamente.')
        inds=np.unique(np.round(np.linspace(0,len(common)-1,n_fields)).astype(int));chosen=[common[i] for i in inds]
        if n_fields==1:chosen=[common[0]]
        parts.append(t[t.campo.isin(chosen)])
        audit.append({'site':s,'placa':p,'replica':r,'campos_disponiveis_em_comum':','.join(common),'campos_escolhidos':','.join(chosen),'criterio':'primeiro, ou indices igualmente espacados na lista natural; independente da imagem'})
    selected=pd.concat(parts,ignore_index=True).merge(plan,on=['site','placa','replica','poco'],validate='many_to_one')
    return selected,pd.DataFrame(audit)

def prepare_download(data,gallery,out,max_total_mb=1024,max_file_mb=128):
    rows=[];cache={}
    for k in data.key.unique():
        info=gallery.head(k);require(info['bytes']<=max_file_mb*1024**2,'Imagem excede limite individual')
        cache[k]=info
    for _,r in data.iterrows(): rows.append({**r.to_dict(),**cache[r.key]})
    result=pd.DataFrame(rows)
    total=sum(v['bytes'] for v in cache.values())
    freeze_table(Path(out)/'manifesto_imagens.csv',result)
    jsave(Path(out)/'tamanho_previsto.json',{'n_objetos_unicos':len(cache),'bytes':total,'limite_MB':max_total_mb})
    require(total<=max_total_mb*1024**2,f'Plano exige {total/1024**2:.1f} MiB; limite {max_total_mb}. Rever volume antes de baixar.')
    return result

def read_plane(path):
    path=Path(path)
    if path.suffix.lower()=='.png':
        with Image.open(path) as im: a=np.asarray(im)
    else:
        with tifffile.TiffFile(path) as tf:
            require(len(tf.pages)==1, 'TIFF com mais de um plano; nao escolher/projetar automaticamente: '+str(path))
            page=tf.pages[0]
            try:a=page.asarray()
            except ValueError as exc:
                if 'imagecodecs' not in str(exc):raise
                with Image.open(path) as im:
                    require(getattr(im,'n_frames',1)==1, 'TIFF multipagina no fallback')
                    a=np.array(im)
                require(a.shape==page.shape and a.dtype==np.dtype(page.dtype), 'Fallback alterou forma ou tipo dos pixels')
    require(a.ndim==2, 'Imagem nao e plano 2D: '+str(path))
    require(np.isfinite(a).all(), 'Pixels nao finitos: '+str(path))
    return a

def reference_limits(arrays,q=(0.5,99.5)):
    # Equal deterministic sampling per control field; display only, no preprocessing of profiles.
    v=[]
    for a in arrays:
        f=np.asarray(a).ravel();n=min(len(f),100000)
        v.append(f[np.linspace(0,len(f)-1,n,dtype=int)])
    require(v,'Sem DMSO para escala de exibicao')
    lo,hi=np.percentile(np.concatenate(v),q)
    require(np.isfinite(lo+hi) and hi>lo,'DMSO com faixa nula; nao ajustar pelos tratamentos')
    return float(lo),float(hi)

def display_plane(a,lo,hi):
    require(hi>lo,'Limites de contraste invalidos')
    b=(np.clip((a.astype(float)-lo)/(hi-lo),0,1)*255).round().astype('uint8')
    return Image.fromarray(b),float(np.mean(a<lo)),float(np.mean(a>hi))


# ----------------------- Protocol and file preservation -----------------------
VERSION='alda1-replicas-1.0'
SOURCE_SHA256='4128c1ee872ba5080264ad9d1db2cacaf34425f93220d99d8a46ccc63bda46a1'
CONTROL_SEED='20261003'
WELL_KEYS=['site','placa','replica','poco']
IMAGE_KEYS=WELL_KEYS+['campo','canal']

def stable_json(value):
    return json.dumps(value,ensure_ascii=False,sort_keys=True,default=str).encode('utf-8')

def freeze_json(path,value):
    path=Path(path)
    if path.exists():
        require(json.loads(path.read_text())==value, 'Conteudo congelado mudou: '+str(path))
    else: jsave(path,value)
    return digest(path)

def freeze_table(path,frame):
    path=Path(path); payload=frame.to_csv(index=False).encode('utf-8')
    if path.exists():
        require(path.read_bytes()==payload, 'Tabela congelada mudou: '+str(path))
    else:path.write_bytes(payload)
    return digest(path)

def validate_dev_plan(plan,sources):
    require(set(plan.site)==set(SITES), 'Esperados os quatro locais')
    require(set(plan.replica)==set(REPS), 'Somente R1-R3 no desenvolvimento')
    require(not plan.duplicated(WELL_KEYS).any(), 'Pocos duplicados')
    for (site,rep),g in plan.groupby(['site','replica']):
        require(len(g)==3 and g.composto.eq('EOS100315').sum()==1 and g.composto.eq('DMSO').sum()==2,
                'Esperados Alda-1 e dois DMSO: '+str((site,rep)))
    require(len(plan)==36, 'Plano de Alda-1 incompleto')
    for site,g in plan.groupby('site'):
        m=pd.read_csv(Path(sources)/f'mapa_{site}_R1_R3.csv').rename(columns={
            'Metadata_Plate':'placa','Metadata_Batch':'replica','Metadata_Well':'poco',
            'Metadata_EOS':'composto','Metadata_Concentration':'dose'})
        m['poco']=m.poco.map(normwell)
        q=g.merge(m[['placa','replica','poco','composto','dose']],on=['placa','replica','poco'],
                  how='left',validate='one_to_one',suffixes=('_plano','_mapa'))
        require(q.composto_plano.eq(q.composto_mapa).all(),'Identidade difere no mapa de '+site)
        require(np.array_equal(q.dose_plano.astype(float),q.dose_mapa.astype(float)),'Dose difere no mapa de '+site)
    return plan

def initialize_campaign(root,source_zip,code_hash,versions,resume=''):
    root,source_zip=Path(root),Path(source_zip)
    require(source_zip.is_file() and digest(source_zip)==SOURCE_SHA256,'Pacote documental ausente/alterado')
    config=dict(version=VERSION,code_sha256=code_hash,source_sha256=SOURCE_SHA256,
        cases=['EOS100315'],sites=SITES,replicas=REPS,n_DMSO=2,seed_controls=CONTROL_SEED,
        n_fields=3,field_selection='indices igualmente espacados na intersecao R1-R3 de todos os pocos por site/placa',
        channels=CHANNELS,display_DMSO_quantiles=[0.5,99.5],
        supplemental_upper='max(DMSO_hi, P99.5 de cada campo de tratamento e controles da placa/replica/canal)',
        supplemental_channels=CHANNELS,crop_center_pixels=512,
        no_segmentation=True,no_new_score=True,no_models=True,R4_in_development=False)
    if resume:
        run=Path(resume)
        require(run.is_dir() and (run/'protocolo.json').is_file(),'Pasta de retomada nao e uma campanha deste caderno')
    else:
        run=root/'resultados'/('Alda1_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
        run.mkdir(parents=True,exist_ok=False)
    freeze_json(run/'protocolo.json',config)
    # Refuse a changed environment within one run, rather than silently overwrite it.
    freeze_json(run/'versoes.json',versions)
    sources=run/'fontes'
    if not sources.exists(): safe_extract(source_zip,sources)
    for name,h in json.loads((sources/'manifesto_arquivos.json').read_text()).items():
        require(digest(sources/name)==h,'Fonte local modificada: '+name)
    dev=run/'R1_R3';dev.mkdir(exist_ok=True)
    plan=validate_dev_plan(make_plan(sources,'completo',codes=['EOS100315'],replicas=REPS),sources)
    freeze_table(dev/'plano_pocos.csv',plan)
    measures=pd.read_csv(sources/'medidas_casos_com_sinal.csv')
    measures=measures[measures.composto.eq('EOS100315')].copy()
    freeze_table(dev/'medidas_Alda1_com_sinal_R1_R3.csv',measures)
    return run,sources,plan

# ----------------------- Comparable field sampling -----------------------
def choose_common_fields(data,plan,n_fields=3,locked=None):
    """IDs are sampling labels, not co-registered spatial coordinates between sites."""
    require(n_fields>=1,'Numero de campos invalido')
    require(not data.duplicated(IMAGE_KEYS).any(),'Imagens documentais duplicadas')
    data=data.copy();data['campo']=data.campo.map(normfield)
    parts=[];audit=[]
    for (site,plate),g in plan.groupby(['site','placa']):
        t=data[(data.site==site)&(data.placa==plate)]
        sets=[]
        for _,r in g.iterrows():
            z=t[(t.replica==r.replica)&(t.poco==r.poco)]
            fields=z.groupby('campo').canal.agg(set)
            fs=set(fields[fields.map(lambda c:c==set(CHANNELS))].index)
            require(fs,'Nenhum campo com todos os canais: '+str((site,plate,r.replica,r.poco)))
            sets.append(fs)
        common=sorted(set.intersection(*sets),key=natural)
        if locked is not None:
            lock=locked[(locked.site==site)&(locked.placa==plate)]
            require(len(lock)==1,'Selecao de campos de desenvolvimento ausente/ambigua')
            chosen=[normfield(f) for f in str(lock.iloc[0].campos_escolhidos).split(',')]
            require(set(chosen)<=set(common),'R4 nao possui os campos congelados; nao substituir por outros')
        else:
            require(len(common)>=n_fields,'Menos campos comuns que o planejado: '+str((site,plate)))
            ix=np.unique(np.round(np.linspace(0,len(common)-1,n_fields)).astype(int))
            chosen=[common[i] for i in ix]
        require(len(chosen)==n_fields,'Quantidade de campos diferente do protocolo')
        pieces=t[t.campo.isin(chosen)].merge(g,on=WELL_KEYS,how='inner',validate='many_to_one')
        require(len(pieces)==len(g)*n_fields*len(CHANNELS),'Selecao incompleta de imagens')
        parts.append(pieces)
        audit.append(dict(site=site,placa=plate,replicas=','.join(sorted(g.replica.unique())),
          campos_disponiveis_em_comum=','.join(common),campos_escolhidos=','.join(chosen),
          regra='campos congelados de R1-R3' if locked is not None else 'indices igualmente espacados em R1-R3, independente de aparencia'))
    return pd.concat(parts,ignore_index=True),pd.DataFrame(audit)

# ----------------------- Downloads, reuse and integrity -----------------------
def cached_image(key,info,folders):
    suffix=Path(key).suffix.lower()
    suffix=suffix if suffix in ['.tif','.tiff','.png'] else '.dat'
    name=hashlib.sha256(key.encode()).hexdigest()+suffix
    for folder in folders:
        path=Path(folder)/name;sidecar=path.with_suffix(path.suffix+'.json')
        if not path.exists():continue
        require(sidecar.is_file(),'Imagem em cache sem registro de integridade: '+str(path))
        saved=json.loads(sidecar.read_text())
        require(saved.get('key')==key,'Chave divergente no cache')
        require(saved['bytes']==path.stat().st_size and digest(path)==saved['sha256'],'Imagem de cache modificada')
        if saved['etag']!=str(info['etag']) or saved['bytes']!=int(info['bytes']):
            continue  # Valid old version; keep it but do not reuse for a changed remote object.
        return path,saved
    return None

def planned_downloads(selected,gallery,out,total_mib=2048,cache_folders=()):
    out=Path(out)
    manifest=prepare_download(selected,gallery,out,max_total_mb=total_mib)
    reused={}
    for key,g in manifest.groupby('key'):
        found=cached_image(key,g.iloc[0],cache_folders)
        if found:reused[key]=(str(found[0]),found[1]['sha256'])
    summary=dict(n_images=len(manifest),n_unique=manifest.key.nunique(),
        n_reutilizaveis=len(reused),bytes_total=int(manifest.drop_duplicates('key').bytes.sum()),
        bytes_novos=int(manifest.drop_duplicates('key').loc[lambda x:~x.key.isin(reused),'bytes'].sum()),
        limite_MiB=total_mib)
    jsave(out/'volume_download.json',summary)
    return manifest,summary

def download_images_cached(manifest,gallery,out,cache_folders=(),before_first_access=None):
    out=Path(out);dest=out/'imagens_originais';dest.mkdir(exist_ok=True)
    rows=[];notified=False
    for key,g in manifest.groupby('key',sort=True):
        info=g.iloc[0]
        if not notified and before_first_access is not None:
            before_first_access();notified=True
        found=cached_image(key,info,[dest]+list(cache_folders))
        if found:
            path,saved=found;origin='cache_integridade_conferida'
        else:
            suffix=Path(key).suffix.lower();suffix=suffix if suffix in ['.tif','.tiff','.png'] else '.dat'
            path=dest/(hashlib.sha256(key.encode()).hexdigest()+suffix)
            # A stale file inside this campaign is not silently replaced.
            require(not path.exists(),'Objeto remoto mudou dentro da campanha; iniciar nova execucao')
            tmp=path.with_suffix(path.suffix+'.part');n=0
            try:
                with gallery.request('GET',object_url(key),stream=True) as response:
                    require(response.headers.get('ETag','')==str(info.etag),'ETag mudou desde o manifesto')
                    with tmp.open('wb') as stream:
                        for part in response.iter_content(1024*1024):
                            n+=len(part);require(n<=int(info.bytes),'Transferencia excedeu tamanho previsto');stream.write(part)
                require(n==int(info.bytes),'Transferencia incompleta')
                tmp.replace(path)
                saved=dict(key=key,sha256=digest(path),etag=str(info.etag),bytes=n,utc=datetime.now(timezone.utc).isoformat())
                jsave(path.with_suffix(path.suffix+'.json'),saved);origin='download_nesta_campanha'
            finally:
                if tmp.exists():tmp.unlink()
        rows.append(dict(key=key,arquivo_local=str(path),sha256=saved['sha256'],origem_arquivo=origin))
        # Checkpoint after each validated object; crash never implies complete output.
        pd.DataFrame(rows).to_csv(out/'download_checkpoint.csv',index=False)
    result=manifest.merge(pd.DataFrame(rows),on='key',validate='many_to_one')
    result.to_csv(out/'imagens_baixadas.csv',index=False)
    return result

# ----------------------- Display: two shared windows, never auto per image -----------------------
def font_for(size=16):
    try:return ImageFont.truetype('DejaVuSans.ttf',size)
    except OSError:return ImageFont.load_default(size=size)

def make_tile(array,lo,hi,crop=False,tile=384):
    hh,ww=array.shape
    if crop:
        size=min(512,hh,ww);y=(hh-size)//2;x=(ww-size)//2
        array=array[y:y+size,x:x+size]
    image,_,_=display_plane(array,lo,hi)
    image.thumbnail((tile,tile),Image.Resampling.LANCZOS)
    return image

def montage(group,arrays,limits,stage,field,out,crop=False,comparison=False):
    ids=sorted(group.sample_id.unique());site,plate,rep=group.iloc[0][['site','placa','replica']]
    # multichannel: four columns, one row per well; dual Mito: one column per well and two windows.
    tile=384;cw=tile+16;rh=tile+70;header=100
    cols=len(ids) if comparison else len(CHANNELS);nrows=2 if comparison else len(ids)
    canvas=Image.new('RGB',(cols*cw,header+nrows*rh),(255,255,255));dr=ImageDraw.Draw(canvas)
    f=font_for(16);small=font_for(13)
    title=f'{site} {plate} {rep} | campo {field} | '+('recorte central <=512 px' if crop else 'campo inteiro reduzido')
    dr.text((10,8),title,fill=(0,0,0),font=f)
    dr.text((10,32),'IDs mascarados. Uma faixa por placa/replica/canal, compartilhada entre campos e pocos.',fill=(0,0,0),font=small)
    dr.text((10,52),'Nao comparar brilho ou tamanho absoluto entre locais/replicas; sem escala fisica presumida.',fill=(0,0,0),font=small)
    dr.text((10,72),'Linha 1: DMSO. Linha 2: complementar.' if comparison else 'Janela DMSO; arquivos originais sem alteracao.',fill=(0,0,0),font=small)
    for i,sid in enumerate(ids):
        for j,ch in enumerate(['Mito'] if comparison else CHANNELS):
            rr=group[(group.sample_id==sid)&(group.canal==ch)]
            require(len(rr)==1,'Imagem ausente/duplicada no painel')
            row=rr.iloc[0];arr=arrays[row.key];win=limits[ch]
            choices=[('DMSO',win[1]),('complementar',win[2])] if comparison else [('DMSO',win[1])]
            for n,(label,hi) in enumerate(choices):
                x=(i if comparison else j)*cw;y=header+(n if comparison else i)*rh
                dr.text((x+5,y),f'{sid} | {ch}',fill=(0,0,0),font=f)
                dr.text((x+5,y+22),f'{label}: {win[0]:.0f}-{hi:.0f}; acima {100*np.mean(arr>hi):.2f}%',fill=(0,0,0),font=small)
                dr.text((x+5,y+42),f'{arr.shape[1]}x{arr.shape[0]} px originais',fill=(0,0,0),font=small)
                canvas.paste(make_tile(arr,win[0],hi,crop,tile),(x+5,y+65))
    typ='Mito_duas_faixas' if comparison else 'quatro_canais_DMSO'
    view='crop' if crop else 'inteiro'
    name=f'{site}_{plate}_{rep}_campo{field}_{typ}_{view}.png'
    path=Path(out)/'paineis'/name;canvas.save(path)
    return dict(site=site,placa=plate,replica=rep,campo=field,tipo=typ,visao=view,arquivo='paineis/'+name)

def render_phase(manifest,out):
    out=Path(out);(out/'paineis').mkdir(exist_ok=True)
    require(not manifest.duplicated(IMAGE_KEYS).any(),'Imagens repetidas')
    for path,h in manifest[['arquivo_local','sha256']].drop_duplicates().itertuples(index=False,name=None):
        require(digest(path)==h,'TIFF foi alterado antes da exibicao: '+str(path))
    scales=[];diagnostics=[];reviews=[];panels=[]
    for (site,plate,rep),g in manifest.groupby(['site','placa','replica']):
        arrays={r.key:read_plane(r.arquivo_local) for _,r in g.iterrows()}
        require(len({str(a.dtype) for a in arrays.values()})==1,'Tipos de pixel diferentes no mesmo grupo')
        require(len({a.shape for a in arrays.values()})==1,'Dimensoes diferentes; nao redimensionar para comparacao automaticamente')
        limits={}
        for ch in CHANNELS:
            ctrl=g[(g.tipo=='controle_DMSO')&(g.canal==ch)]
            require(ctrl.sample_id.nunique()==2,'A escala exige exatamente dois DMSO')
            lo,hi=reference_limits([arrays[k] for k in ctrl.key])
            # Same rule for every site and channel. Per-field P99.5 contributes only to the ONE shared maximum.
            upper=max(hi,*(float(np.percentile(arrays[k],99.5)) for k in g[g.canal==ch].key))
            limits[ch]=(lo,hi,upper)
            scales.append(dict(site=site,placa=plate,replica=rep,canal=ch,limite_inferior=lo,
                max_DMSO=hi,max_complementar=upper,n_campos_controle=len(ctrl),
                n_pocos_controle=ctrl.sample_id.nunique(),n_campos_total=int(g.canal.eq(ch).sum()),
                regra='DMSO P0.5/P99.5, max complementar comum = maior P99.5 por campo; todos os campos selecionados'))
        for _,r in g.iterrows():
            a=arrays[r.key];lo,hi,up=limits[r.canal];maxdt=np.iinfo(a.dtype).max if a.dtype.kind in 'iu' else np.nan
            diagnostics.append(dict(sample_id=r.sample_id,site=site,placa=plate,replica=rep,campo=r.campo,canal=r.canal,
                altura_px=a.shape[0],largura_px=a.shape[1],dtype=str(a.dtype),
                clip_DMSO_alto_pct=100*float(np.mean(a>hi)),clip_complementar_alto_pct=100*float(np.mean(a>up)),
                clip_baixo_pct=100*float(np.mean(a<lo)),max_dtype_pct=100*float(np.mean(a==maxdt)),
                mediana_pixel=float(np.median(a)),P99_5=float(np.percentile(a,99.5)),sha256=r.sha256,
                unidade='pixels do campo inteiro; nao porcentagem de celulas nem saturacao instrumental'))
        for field,h in g.groupby('campo',sort=True):
            for crop in [False,True]:
                for comparison in [False,True]:panels.append(montage(h,arrays,limits,rep,field,out,crop,comparison))
            for sid in sorted(h.sample_id.unique()):
                reviews.append(dict(sample_id=sid,site=site,placa=plate,replica=rep,campo=field,
                  foco='',fundo_nao_uniforme='',densidade_visual='',detritos='',
                  Mito_brilho_relativo='',Mito_pontual_tubular_ou_indeterminado='',Mito_distribuicao='',
                  efeito_da_janela='',avaliavel_visualmente='',observacoes='',revisor='',data_revisao=''))
        del arrays
    pd.DataFrame(scales).to_csv(out/'escalas_exibicao.csv',index=False)
    pd.DataFrame(diagnostics).to_csv(out/'diagnosticos_pixels.csv',index=False)
    pd.DataFrame(panels).to_csv(out/'indice_paineis.csv',index=False)
    template=pd.DataFrame(reviews)
    template.to_csv(out/'avaliacao_visual_modelo.csv',index=False)
    human=out/'avaliacao_visual_PREENCHER.csv'
    if human.exists():
        prior=pd.read_csv(human,keep_default_na=False,dtype={'campo':str})
        k=['sample_id','campo']
        require(set(map(tuple,prior[k].astype(str).values))==set(map(tuple,template[k].astype(str).values)),
                'Ficha humana tem campos diferentes; nao sobrescrever')
    else:template.to_csv(human,index=False)
    manifest[['sample_id']+WELL_KEYS+['composto','nome','tipo']].drop_duplicates().to_csv(out/'chave_identidades_ABRIR_DEPOIS.csv',index=False)
    import html
    lines=['<!doctype html><html lang="pt"><meta charset="utf-8"><title>Paineis Alda-1</title>',
           '<body style="font-family:Arial;max-width:1100px;margin:auto"><h1>Inspecao Alda-1</h1>',
           '<p>Imagens originais; campos repetidos nao sao replicas independentes. Use a ficha antes da chave.</p>']
    for p in panels:
        name=html.escape(p['arquivo']);label=html.escape(' | '.join(str(p[k]) for k in ['site','replica','campo','tipo','visao']))
        lines.append(f'<details><summary>{label}</summary><a href="{name}">Abrir em tamanho maior</a><br><img loading="lazy" style="max-width:100%" src="{name}"></details>')
    lines.append('</body></html>');(out/'indice_paineis.html').write_text('\n'.join(lines),encoding='utf-8')
    # Recheck original arrays' files after rendering.
    for path,h in manifest[['arquivo_local','sha256']].drop_duplicates().itertuples(index=False,name=None):
        require(digest(path)==h,'Arquivo original mudou durante a exibicao')
    return pd.DataFrame(panels)

# ----------------------- Optional R4; never touched by the default branch -----------------------
def R4_sites(authorize=False,include_medina=False,medina_ack=''):
    if not authorize:return []
    if include_medina:
        require(medina_ack=='AUTORIZO INSPECAO VISUAL MEDINA R4',
                'Para incluir MEDINA, registre a autorizacao explicita. A imagem deixara de ser nao consultada.')
    return SITES if include_medina else ['IMTM','FMP','USC']

def prepare_R4_plan(dev_plan,gallery,out,sites):
    require(sites and set(sites)<=set(SITES),'Locais R4 nao autorizados')
    require(set(dev_plan.replica)<=set(REPS),'Plano de desenvolvimento invalido')
    rows=[];audit=[]
    for site in sites:
        cases=dev_plan[(dev_plan.site==site)&(dev_plan.tipo!='controle_DMSO')][['placa','composto','dose','nome']].drop_duplicates()
        aliases,records=barcode_lookup(gallery,site)
        for plate,q in cases.groupby('placa'):
            keys={r['platemap_key'] for r in records if r['placa']==plate and r['replica']=='R4' and r.get('platemap_key')}
            require(len(keys)==1,'Platemap R4 ausente/ambiguo para '+site+'/'+plate)
            key=next(iter(keys));blob=gallery.bytes(key);m=read_remote_table(blob)
            def one(choices):
                found=[c for c in choices if c in m];require(len(found)==1,'Coluna ambigua/ausente no mapa R4');return found[0]
            wc=one(['well_position','Metadata_Well']);ec=one(['Metadata_EOS','compound_id']);dc=one(['Metadata_Concentration','Concentration'])
            m=m.copy();m['poco']=m[wc].map(normwell);m['composto']=m[ec].astype(str).str.strip();m['dose']=pd.to_numeric(m[dc],errors='coerce')
            require(m.poco.is_unique,'Pocos repetidos no mapa R4')
            wanted=[]
            for _,case in q.iterrows():
                hit=m[(m.composto==case.composto)&(m.dose==float(case.dose))]
                require(len(hit)==1,'Alda-1/dose nao localizado unicamente em R4; nao copiar endereco de R1')
                record=hit.iloc[0];wanted.append((record,case['nome'],'caso_biologico'))
            controls=m[m.composto.eq('DMSO')].copy();require(len(controls)>=2,'Menos de dois DMSO em R4')
            controls['order']=controls.poco.map(lambda w:hashlib.sha256(f'{CONTROL_SEED}|{site}|{plate}|R4|{w}'.encode()).hexdigest())
            for _,r in controls.sort_values('order').head(2).iterrows():wanted.append((r,'DMSO','controle_DMSO'))
            for r,name,typ in wanted:
                require(np.isfinite(float(r.dose)),'Dose de controle nao documentada')
                sid='S'+hashlib.sha256(f'{site}|{plate}|R4|{r.poco}'.encode()).hexdigest()[:12]
                rows.append(dict(site=site,placa=plate,replica='R4',poco=r.poco,composto=r.composto,dose=float(r.dose),
                    nome=name,tipo=typ,sample_id=sid,n_objetos=np.nan,A_Mito=np.nan,
                    origem='platemap publico R4; sem copiar escores/contagens de R1-R3',
                    criterio_controles='2 DMSO por SHA256; mesma regra do desenvolvimento'))
            audit.append(dict(site=site,placa=plate,replica='R4',key=key,sha256=hashlib.sha256(blob).hexdigest()))
    plan=pd.DataFrame(rows).sort_values(['site','placa','replica','sample_id']).reset_index(drop=True)
    require(not plan.duplicated(WELL_KEYS).any(),'R4 tem duplicacao')
    require(len(plan)==len(sites)*3,'Numero inesperado de pocos R4')
    freeze_table(Path(out)/'mapas_R4_fontes.csv',pd.DataFrame(audit))
    freeze_table(Path(out)/'plano_pocos.csv',plan)
    return plan

def exposure_event(run,sites,stage):
    path=Path(run)/'registro_acesso_R4.jsonl'
    event=dict(utc=datetime.now(timezone.utc).isoformat(),sites=sites,etapa=stage,
        MEDINA_nova_inspecao='MEDINA' in sites,
        escopo='Alda-1 e DMSO; revisao visual descritiva posterior, nao triagem ou validacao funcional')
    with path.open('a',encoding='utf-8') as f:f.write(json.dumps(event,ensure_ascii=False)+'\n')

def close_phase(run,phase,plan,manifest,panels):
    run,phase=Path(run),Path(phase);human=pd.read_csv(phase/'avaliacao_visual_PREENCHER.csv',keep_default_na=False)
    n_done=int(human.avaliavel_visualmente.astype(str).str.strip().ne('').sum())
    r4='R4' in set(plan.replica)
    status=dict(phase=phase.name,replicas=sorted(plan.replica.unique()),sites=sorted(plan.site.unique()),
        n_wells=len(plan),n_images=manifest.key.nunique(),n_fields_per_well=3,n_panels=len(panels),
        n_forms=len(human),n_forms_with_evaluability=n_done,R4_read_in_this_phase=r4,
        models_retrained=False,scores_changed=False,candidates_reclassified=False,segmentation=False)
    jsave(phase/'estado.json',status)
    report=f'''# Inspecao de Alda-1: {phase.name}

Pocos: {len(plan)}. Imagens: {manifest.key.nunique()}. Paineis: {len(panels)}.
Fichas por campo: {len(human)}; avaliabilidade preenchida: {n_done}.
Locais: {', '.join(sorted(plan.site.unique()))}. Replicas: {', '.join(sorted(plan.replica.unique()))}.

Janela primaria estimada em dois DMSO e todos os campos selecionados por placa/replica/canal.
Janela complementar: mesmo limite inferior e maximo dos P99.5 por campo, comum a todos os pocos e campos desse grupo.
Regras de exibicao iguais nos quatro locais; valores numericos diferentes entre placas e replicas.
A janela complementar nao e correcao de iluminacao, normalizacao de perfis ou analise funcional.
Clipping usa pixels do campo inteiro; limite uint16 nao determina a saturacao do detector.
Os tres campos sao subamostras do mesmo poco, nao tres experimentos independentes.
Amostras foram escolhidas antes das imagens desta ampliacao; Alda-1 e caso de estudo pos-hoc, nao nova descoberta.

R4: {'inspecao separada, descritiva e registrada; sem recalculo da regra' if r4 else 'nao acessada nesta fase'}.
Nenhum modelo, SHAP, escore ou candidato foi alterado. A ficha humana nao foi preenchida automaticamente.
As imagens nao demonstram atividade de ALDH2, massa, potencial de membrana, fissao, fusao ou toxicidade.
'''
    (phase/'relatorio_inspecao.md').write_text(report,encoding='utf-8')
    # Completion binds the source protocol, field plan and downloaded-object identity, not the human's later annotations.
    stamp=dict(protocolo_sha256=digest(run/'protocolo.json'),plano_sha256=digest(phase/'plano_pocos.csv'),
       campos_sha256=digest(phase/'campos_escolhidos.csv'),manifesto_sha256=digest(phase/'imagens_baixadas.csv'))
    freeze_json(phase/'conclusao_congelada.json',stamp)
    package=phase.with_name(phase.name+'_leitura.zip')
    with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
        for p in phase.rglob('*'):
            if p.is_file() and not {'imagens_originais','cache_metadados'}.intersection(p.relative_to(phase).parts):
                z.write(p,p.relative_to(phase))
        for p in [run/'protocolo.json',run/'versoes.json']:
            z.write(p,'campanha/'+p.name)
    return report,package

def check_development_complete(run):
    run=Path(run);dev=run/'R1_R3';p=dev/'conclusao_congelada.json'
    require(p.is_file(),'Finalize e congele R1-R3 antes de autorizar R4')
    saved=json.loads(p.read_text())
    checks={'protocolo_sha256':run/'protocolo.json','plano_sha256':dev/'plano_pocos.csv',
            'campos_sha256':dev/'campos_escolhidos.csv','manifesto_sha256':dev/'imagens_baixadas.csv'}
    for k,path in checks.items():require(digest(path)==saved[k],'Desenvolvimento alterado depois do fechamento')
    return dev

def run_optional_R4(run,gallery,authorize=False,include_medina=False,medina_ack='',limit_mib=1024,cache_folders=()):
    sites=R4_sites(authorize,include_medina,medina_ack)
    if not sites:return None  # Crucial: no file read, metadata query, HEAD or pixel access.
    dev=check_development_complete(run)
    out=Path(run)/('R4_IMTM_FMP_USC_MEDINA' if include_medina else 'R4_IMTM_FMP_USC')
    out.mkdir(exist_ok=True)
    consent=dict(sites=sites,medina_ack=medina_ack if include_medina else '',
        desenvolvimento=digest(dev/'conclusao_congelada.json'),
        natureza='inspecao visual pos-hoc; R4 dos outros locais ja consultada nas analises anteriores')
    freeze_json(out/'autorizacao.json',consent)
    exposure_event(run,sites,'consulta_metadados_autorizada_sem_pixels')
    plan=prepare_R4_plan(pd.read_csv(dev/'plano_pocos.csv'),gallery,out,sites)
    data=discover_images(plan,gallery,out,allowed_reps=('R4',))
    data.to_csv(out/'imagens_disponiveis_pocos.csv',index=False)
    selected,fields=choose_common_fields(data,plan,3,locked=pd.read_csv(dev/'campos_escolhidos.csv',dtype=str))
    freeze_table(out/'campos_escolhidos.csv',fields)
    manifest,volume=planned_downloads(selected,gallery,out,limit_mib,[out/'imagens_originais']+list(cache_folders))
    print('R4, volume planejado:',volume)
    images=download_images_cached(manifest,gallery,out,cache_folders,
        before_first_access=lambda:exposure_event(run,sites,'acesso_a_pixels_iniciado'))
    panels=render_phase(images,out)
    exposure_event(run,sites,'paineis_gerados')
    report,package=close_phase(run,out,plan,images,panels)
    return dict(output=out,report=report,package=package)


## 4. Abrir uma campanha sem sobrescrever o piloto

As fontes R1–R3 são as mesmas da preparação anterior, verificadas por integridade. A escolha dos controles usa o mesmo hash do piloto; por isso, os controles de R1 são mantidos. Cada local e réplica tem seus próprios DMSO.

Os parâmetros, a fonte e o código realmente executado são registrados. Se houver uma mudança durante a retomada, use uma campanha nova em vez de misturar versões.

In [ ]:
# Registrar o texto da celula de funcoes executada nesta sessao.
HISTORICO = get_ipython().history_manager.input_hist_raw
BLOCOS = [s for s in HISTORICO if s.lstrip().startswith('"""Alda-1 visual follow-up.')]
if not BLOCOS:
    raise RuntimeError('Execute primeiro a celula completa da secao 3.')
CODIGO_EXECUTADO = BLOCOS[-1]
HASH_CODIGO = hashlib.sha256(CODIGO_EXECUTADO.encode('utf-8')).hexdigest()
CAMPANHA, FONTES, PLANO = initialize_campaign(RAIZ, FONTES_ZIP, HASH_CODIGO, VERSOES, RETOMAR_PASTA)
(CAMPANHA / 'codigo_executado.py').write_text(CODIGO_EXECUTADO, encoding='utf-8')
SAIDA = CAMPANHA / 'R1_R3'
GALERIA = PublicGallery(SAIDA / 'cache_metadados')
print('CAMPANHA:', CAMPANHA)
display(PLANO.groupby(['site','replica','tipo']).size().rename('n_pocos').reset_index())


## 5. Confirmar poços, campos e canais documentais

O plano inclui todas as três réplicas, mesmo que uma delas tenha escore menor ou imagem menos favorável no piloto. A consulta abaixo usa apenas as combinações solicitadas de R1–R3. Tabelas gerais de barcodes podem mencionar R4, mas nenhum `load_data`, poço quantitativo ou imagem de R4 entra nesta fase.

A dose numérica é comparada à fonte, sem corrigir valores ou deduzir unidades pela aparência do nome do arquivo. Falhas ficam registradas; não são convertidas em ausência de resposta biológica.

In [ ]:
try:
    DISPONIVEIS = discover_images(PLANO, GALERIA, SAIDA, allowed_reps=('R1','R2','R3'))
    freeze_table(SAIDA / 'imagens_disponiveis_pocos.csv', DISPONIVEIS)
    display(DISPONIVEIS.groupby(['site','replica']).agg(pocos=('poco','nunique'),campos=('campo','nunique'),canais=('canal','nunique')).reset_index())
except Exception as erro:
    jsave(SAIDA / 'falha_acesso.json', {'erro':str(erro),'R4_acessada':False})
    raise


## 6. Congelar os campos e o volume

Selecionamos três IDs igualmente espaçados na lista de campos completos, comuns a **todos os poços e a R1–R3 de cada local/placa**. Para nove campos numerados de 1 a 9, a regra escolhe 1, 5 e 9. O piloto documentou esses nove campos em R1; R2 e R3 ainda precisam passar por esta conferência.

Os IDs iguais não demonstram a mesma posição física em microscópios diferentes. Se faltarem campos, o caderno não escolhe outros com melhor aparência. O plano para antes do download. A retomada também não pode trocar silenciosamente o manifesto.

In [ ]:
SELECIONADAS, CAMPOS = choose_common_fields(DISPONIVEIS, PLANO, n_fields=3)
freeze_table(SAIDA / 'campos_escolhidos.csv', CAMPOS)
MANIFESTO, VOLUME = planned_downloads(SELECIONADAS, GALERIA, SAIDA,
    total_mib=LIMITE_R1_R3_MiB, cache_folders=[SAIDA/'imagens_originais'] + PASTAS_CACHE)
display(CAMPOS)
print(json.dumps(VOLUME, indent=2))
print('MiB totais:', round(VOLUME['bytes_total']/1024**2,1), '| MiB novos:', round(VOLUME['bytes_novos']/1024**2,1))


## 7. Baixar ou reutilizar somente os arquivos planejados

A cada arquivo, o registro salva a origem, o tamanho e o hash. Uma interrupção deixa um checkpoint; arquivos já validados são reaproveitados. Se um arquivo local tiver sido modificado, o procedimento interrompe em vez de substituí-lo automaticamente.

In [ ]:
BAIXADAS = download_images_cached(MANIFESTO, GALERIA, SAIDA, cache_folders=PASTAS_CACHE)
print('Imagens conferidas:', BAIXADAS.key.nunique())
display(BAIXADAS.groupby('origem_arquivo').size().rename('n_arquivos').reset_index())


## 8. Painéis e auditoria dos pixels

Há duas perguntas de visualização. A janela DMSO mostra o sinal relativo aos controles; uma janela complementar permite examinar detalhes que ultrapassam aquela faixa.

**Janela principal:** percentis 0,5 e 99,5 dos pixels amostrados nos dois DMSO, incluindo seus três campos. **Complementar:** o limite inferior é mantido, e o superior é o maior entre o limite DMSO e o percentil 99,5 de cada campo selecionado do tratamento e dos controles. Uma única faixa por placa/réplica/canal é aplicada a todos os seus campos. Não existe autoajuste por imagem.

A regra complementar é explicitamente baseada no diagnóstico do piloto e foi fixada **antes** desta ampliação. As faixas numéricas de R1 podem mudar em relação ao piloto porque agora incluem três campos; os painéis anteriores permanecem preservados.

Para cada campo geram-se quatro painéis: quatro canais na janela DMSO, inteiro e recorte central; e Mito com as duas janelas, inteiro e recorte. O recorte tem até 512 pixels e não é escolhido pela presença de uma célula. Não se presume tamanho físico de pixel. Clipping refere-se aos pixels do campo inteiro e não a porcentagem de células; o teto uint16 não identifica saturção instrumental.

In [ ]:
PAINEIS = render_phase(BAIXADAS, SAIDA)
print('Paineis salvos:', len(PAINEIS))
print('Indice de navegacao:', SAIDA / 'indice_paineis.html')
display(pd.read_csv(SAIDA / 'escalas_exibicao.csv'))


## 9. Navegar sem sobrecarregar o Colab

O conjunto completo pode ter 144 painéis. Em vez de imprimir todos, use a função abaixo para abrir um painel. O arquivo `indice_paineis.html` acompanha o ZIP e funciona ao extrair o pacote, preservando sua pasta `paineis`.

Comece pela ficha mascarada. O estudo não é uma avaliação cega independente: o composto e a pergunta já são conhecidos, mas os IDs evitam colocar o nome do tratamento em cada linha antes do registro.

In [ ]:
def mostrar_painel(site='IMTM', replica='R1', campo='1', tipo='Mito_duas_faixas', visao='crop', pasta=None):
    pasta = Path(pasta) if pasta is not None else SAIDA
    idx = pd.read_csv(pasta / 'indice_paineis.csv', dtype={'campo':str})
    escolhido = idx[(idx.site==site)&(idx.replica==replica)&(idx.campo==str(campo))&(idx.tipo==tipo)&(idx.visao==visao)]
    if len(escolhido)!=1:
        display(idx[['site','replica','campo','tipo','visao']])
        raise ValueError('Painel nao localizado unicamente; consulte a tabela acima.')
    display(IPyImage(filename=str(pasta / escolhido.iloc[0].arquivo)))

# Outros exemplos: replica='R2', campo='5', tipo='quatro_canais_DMSO', visao='inteiro'.
# mostrar_painel('USC', 'R2', '5')
print('Ficha humana:', SAIDA / 'avaliacao_visual_PREENCHER.csv')


## 10. Registrar a leitura por campo e depois por réplica

Preencha foco, fundo, ocupação, detritos, brilho relativo de Mito, aparência pontual/tubular ou indeterminada e influência da janela. Use “não avaliável” quando a imagem não sustentar a observação. Não troque automaticamente pouca ocupação por toxicidade ou maior brilho por maior massa/função mitocondrial.

Os três campos são subamostras do mesmo poço; **não transformam três réplicas em nove experimentos independentes**. Primeiro verifique a consistência entre campos de cada poço, depois entre R1–R3 e somente então entre locais. Os recortes e as duas janelas são modos de ver os mesmos dados, não observações independentes.

A ficha nunca é preenchida automaticamente nem sobrescrita ao repetir a montagem. A chave dos poços fica em `chave_identidades_ABRIR_DEPOIS.csv`. A ausência de células no recorte central não implica ausência no poço.

## 11. Relacionar imagens aos perfis já calculados

As contagens e os escores do plano são agregados por poço. Não foram medidos no campo mostrado. O arquivo de medidas com sinal é uma consulta aos perfis existentes de R1–R3; não foi criado um novo escore com as características comuns.

Uma resposta quantitativa de magnitude pode coexistir com sentidos diferentes das medidas individuais. Compare a pergunta visual aos valores de cada réplica, sem exigir que um campo isolado “confirme” a mediana das três. A observação de intensidade, por si só, não demonstra atividade de ALDH2.

In [ ]:
MEDIDAS = pd.read_csv(SAIDA / 'medidas_Alda1_com_sinal_R1_R3.csv')
EXEMPLO = MEDIDAS[MEDIDAS.medida.eq('Cells_RadialDistribution_MeanFrac_Mito_3of4')]
display(EXEMPLO[['site','medida','z_R1','z_R2','z_R3','mediana_z']])


## 12. Fechar R1–R3

O fechamento registra os arquivos produzidos e o número de fichas preenchidas, sem afirmar que a avaliação humana foi concluída. Ele vincula o protocolo, o plano e os campos antes da eventual abertura de R4. **Zero fichas preenchidas não é erro computacional.**

O pacote de leitura contém os painéis, tabelas e o índice, mas não duplica os TIFFs. Após editar a ficha, execute esta célula novamente para atualizá-la no pacote.

In [ ]:
RELATORIO, PACOTE = close_phase(CAMPANHA, SAIDA, PLANO, BAIXADAS, PAINEIS)
display(Markdown(RELATORIO))
print('RESULTADOS:', SAIDA)
print('PACOTE DE LEITURA:', PACOTE)


## 13. R4 opcional — execute somente após a leitura de R1–R3

A primeira execução deve terminar acima. **Mantenha os controles abaixo desligados para rodar somente R1–R3.** Ativá-los não é necessário para concluir esta ampliação.

R4 de IMTM, FMP e USC já foi consultada nas análises anteriores; a nova leitura é um acompanhamento visual descritivo, não um teste independente reiniciado. MEDINA/R4 ainda não foi examinada e requer um consentimento adicional. Ao abrir suas imagens, não poderemos descrevê-las depois como completamente não consultadas para esta pergunta. O registro histórico original não será alterado: o acesso adicional fica em `registro_acesso_R4.jsonl`.

A etapa escolhe Alda-1 e dois DMSO pelo mapa documental de R4, sem copiar o endereço de R1 ou importar os resultados de outros poços. Mantém **os mesmos IDs de campo selecionados em R1–R3** por local. Não pesquisa novos candidatos, calcula a regra ou ajusta parâmetros. O plano sem MEDINA tem nove poços e 108 imagens; com MEDINA, 12 poços e 144 imagens, sujeito à correspondência dos metadados.

As janelas são calculadas pelos DMSO e pela regra complementar desta réplica, somente para visualização. **Contagens e escores R4 não são copiados de R1–R3 nem preenchidos por suposição.**

In [ ]:
EXECUTAR_R4 = False
INCLUIR_MEDINA_R4 = False
CONFIRMACAO_MEDINA = ''  # Somente para incluir MEDINA: 'AUTORIZO INSPECAO VISUAL MEDINA R4'
LIMITE_R4_MiB = 1024

RESULTADO_R4 = run_optional_R4(CAMPANHA, GALERIA,
    authorize=EXECUTAR_R4, include_medina=INCLUIR_MEDINA_R4,
    medina_ack=CONFIRMACAO_MEDINA, limit_mib=LIMITE_R4_MiB,
    cache_folders=PASTAS_CACHE + [CAMPANHA/'R4_IMTM_FMP_USC/imagens_originais'])
if RESULTADO_R4 is None:
    print('R4 DESATIVADA: nenhum acesso aos seus metadados especificos ou pixels nesta celula.')
else:
    display(Markdown(RESULTADO_R4['report']))
    print('RESULTADOS R4:', RESULTADO_R4['output'])
    print('PACOTE R4:', RESULTADO_R4['package'])


## 14. Alcance e fontes

Este caderno amplia uma inspeção orientada por hipóteses previamente discutidas. Alda-1 foi escolhido para estudo de caso, não como resultado de uma nova triagem independente. A escolha complementar de contraste surgiu do piloto e não altera a inferência numérica original.

Os arquivos brutos do experimento estão no projeto `cpg0036-EU-OS-bioactives`, na Cell Painting Gallery. A correspondência das imagens segue os metadados dos autores. Os índices de fonte e seus hashes são salvos, junto à procedência das quatro campanhas de perfis.

- Autores do conjunto: https://github.com/schmiedc/EU-OS_bioactives
- Organização dos arquivos: https://broadinstitute.github.io/cellpainting-gallery/data_structure.html
- Piloto de origem: `inspecao_20261003T200334_744222Z`.

**Estado da entrega:** plano real R1–R3 conferido; correspondência e campos reais R1 verificados a partir dos metadados do piloto; leitura dos seis TIFFs Mito disponíveis localmente e testes controlados documentados no pacote. Não foram baixadas nesta entrega todas as imagens R2/R3, nem abertas imagens de R4. A listagem remota e o download completo serão verificados na sua execução do Colab. O ambiente local não conseguiu resolver o host S3.

O resultado final exige conferência humana e não demonstra função mitocondrial, fissionamento/fusão ou mecanismo farmacológico. Hipóteses funcionais devem ser separadas do que foi efetivamente observado.